In [1]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP (Colab / Kaggle)
# ============================================================
import os, sys, gc, json, time, random, shutil, zipfile, warnings, subprocess, importlib.util

if importlib.util.find_spec("ultralytics") is None:
    print("Ultralytics not found. Installing...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    except Exception as e:
        raise RuntimeError("Ultralytics install failed. Turn Internet ON or install it first.") from e
else:
    print("Ultralytics already installed.")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from IPython.display import display, Image as IPImage
import ultralytics
from ultralytics import YOLO

warnings.filterwarnings("ignore")

# ---------------- SEED (set once) ----------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------------- GPU ----------------
assert torch.cuda.is_available(), "GPU not available! Enable a GPU in the runtime settings."
n_gpu = torch.cuda.device_count()

MULTI_GPU = True            # set False if DDP hangs in your notebook
if n_gpu >= 2 and MULTI_GPU:
    DEVICE, BATCH = "0,1", 32
else:
    DEVICE, BATCH = "0", 16

VAL_DEVICE = "0"            # validation runs on a single GPU
VAL_BATCH = 16

# ---------------- WORK DIR (Kaggle / Colab / local) ----------------
if Path("/kaggle/working").exists():
    WORK = Path("/kaggle/working")
elif Path("/content").exists():
    WORK = Path("/content")
else:
    WORK = Path.cwd()

# ---------------- INFO ----------------
print("\n========== ENVIRONMENT ==========")
print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA       :", torch.version.cuda)
print("GPU count  :", n_gpu)
for i in range(n_gpu):
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB)")
print("DEVICE:", DEVICE, "| BATCH:", BATCH, "| VAL_DEVICE:", VAL_DEVICE)
print("WORK  :", WORK)
print("\nCell 1 completed successfully!")

Ultralytics not found. Installing...


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings



========== ENVIRONMENT ==========
Python     : 3.13.16
PyTorch    : 2.11.0+cu130
Ultralytics: 8.4.218
CUDA       : 13.0
GPU count  : 2
GPU 0: Tesla T4 (14.6 GB)
GPU 1: Tesla T4 (14.6 GB)
DEVICE: 0,1 | BATCH: 32 | VAL_DEVICE: 0
WORK  : /kaggle/working

Cell 1 completed successfully!


In [2]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================
CLASS_NAMES = ["Crack", "Pothole"]
NUM_CLASSES = len(CLASS_NAMES)

MODEL_WEIGHTS = "yolo11s.pt"     # COCO pretrained (Internet ON)
EPOCHS = 80
IMGSZ = 640
PATIENCE = 100                   # > EPOCHS, so early stopping is off

# ---------- dataset discovery ----------
PATH_HINT = "rdd"                # "All" dataset path must contain this
BD_HINT = "bangladesh"           # Bangladesh dataset path must contain this (lowercase)
ALL_PATH = None                  # optional exact folder for the All dataset, e.g. "/kaggle/input/rdd2022"
BD_PATH = None                   # optional exact folder for the BD dataset

CM_CONF = 0.25                   # confidence threshold for the confusion matrix

# ---------- EXPERIMENT (change only this, then Run All) ----------
# 1: All -> All test
# 2: BD  -> BD test
# 3: All-model -> BD test (no training; needs EXP1_BEST or Exp 1 run in this session)
# 4: All+BD train -> BD test
EXPERIMENT = 1
EXP1_BEST = None                 # Exp 3: e.g. "/kaggle/input/<dataset>/best.pt"

RUN_NAMES = {1: "yolo11s_all_country", 2: "yolo11s_bd",
             3: "yolo11s_all_to_bd_eval", 4: "yolo11s_all_plus_bd"}
assert EXPERIMENT in RUN_NAMES, "EXPERIMENT must be 1, 2, 3 or 4"

PROJECT = WORK / "RDD_Baseline_Run"
RUN_NAME = RUN_NAMES[EXPERIMENT]
RUN_DIR = PROJECT / RUN_NAME
PROJECT.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_WEIGHTS, "| Epochs:", EPOCHS, "| imgsz:", IMGSZ)
print("EXPERIMENT:", EXPERIMENT, "|", RUN_NAME)
print("\n✅ Cell 2 done")

Model: yolo11s.pt | Epochs: 80 | imgsz: 640
EXPERIMENT: 1 | yolo11s_all_country

✅ Cell 2 done


In [3]:
# ============================================================
# CELL 3 (FIXED): DATASET DISCOVERY + CLASS REMAP + SPLIT + BUILD
# Pools: "all" and "bd"
# Output: YOLO_ROOT/<pool>/<split>/{images,labels}
# ============================================================
import os, shutil, random, yaml
import numpy as np
from pathlib import Path
from collections import Counter

RANDOM_SEED = SEED
TRAIN_RATIO, VAL_RATIO = 0.70, 0.20          # test = remainder (only if no official split)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
SPLIT_ALIAS = {"train": "train", "training": "train",
               "val": "valid", "valid": "valid", "validation": "valid",
               "test": "test", "testing": "test"}
SPLITS = ("train", "valid", "test")
IGNORE_DIRS = {".git", "__pycache__", ".ipynb_checkpoints", "node_modules",
               ".venv", "venv", "runs", "wandb", ".cache", "sample_data"}
COORD_TOL = 1e-4
TREAT_MISSING_AS_BACKGROUND = True
DROP_INVALID = True

# ---- settings read safely from Cell 2 (works even if Cell 2 is old) ----
PATH_HINT_ = globals().get("PATH_HINT", "rdd")
BD_HINT_ = globals().get("BD_HINT", "bangladesh")
ALL_PATH_ = globals().get("ALL_PATH", None)
BD_PATH_ = globals().get("BD_PATH", None)
MANUAL_CLASS_MAP_ = globals().get("MANUAL_CLASS_MAP", None)   # e.g. {0:0, 1:0, 2:0, 3:1}

# default for your Kaggle dataset if ALL_PATH was not set
_DEFAULT_ALL = "/kaggle/input/datasets/noor3027/allroaddamagesplitteddataset/AllRoadDamageSplittedDataset"
if not ALL_PATH_ and Path(_DEFAULT_ALL).exists():
    ALL_PATH_ = _DEFAULT_ALL

DATA_LOCAL = WORK / "auto_dataset"
DATA_LOCAL.mkdir(parents=True, exist_ok=True)
YOLO_ROOT = DATA_LOCAL / "yolo_dataset"
SKIP_PATHS = {str(DATA_LOCAL), str(PROJECT), "/content/drive", "/content/sample_data"}

POOLS_NEEDED = {1: ["all"], 2: ["bd"], 3: ["bd"], 4: ["all", "bd"]}[EXPERIMENT]
EFFECTIVE = {
    1: {"train": ["all"], "valid": ["all"], "test": ["all"]},
    2: {"train": ["bd"],  "valid": ["bd"],  "test": ["bd"]},
    3: {"train": [],      "valid": ["bd"],  "test": ["bd"]},
    4: {"train": ["all", "bd"], "valid": ["all", "bd"], "test": ["bd"]},
}[EXPERIMENT]

print("=" * 75)
print("DATASET DISCOVERY | EXPERIMENT", EXPERIMENT, "| pools:", POOLS_NEEDED)
print("ALL_PATH:", ALL_PATH_)
print("BD_PATH :", BD_PATH_)
print("=" * 75)

# ------------------------------------------------------------
# helpers
# ------------------------------------------------------------
def normalize_split(name):
    return SPLIT_ALIAS.get(str(name).lower().strip())

def is_image(p):
    return p.suffix.lower() in IMG_EXT and p.is_file()

def count_images(folder, recursive=True):
    it = Path(folder).rglob("*") if recursive else Path(folder).iterdir()
    return sum(1 for p in it if is_image(p))

def find_child_dir(parent, name_lower):
    try:
        for d in parent.iterdir():
            if d.is_dir() and d.name.lower() == name_lower:
                return d
    except OSError:
        pass
    return None

def find_split_dir(parent, split):
    try:
        for d in parent.iterdir():
            if d.is_dir() and normalize_split(d.name) == split:
                return d
    except OSError:
        pass
    return None

def read_names(yaml_path):
    try:
        with open(yaml_path, "r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f)
        names = cfg.get("names") if isinstance(cfg, dict) else None
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names, key=lambda x: int(x))]
        return [str(n) for n in names] if names else None
    except Exception:
        return None

def find_names_up(start, levels=3):
    p = Path(start)
    for _ in range(levels + 1):
        for fn in ("data.yaml", "dataset.yaml", "data.yml", "dataset.yml"):
            f = p / fn
            if f.is_file():
                n = read_names(f)
                if n:
                    return n
        if p.parent == p:
            break
        p = p.parent
    return None

# ------------------------------------------------------------
# discovery of (images, labels) folder pairs
# ------------------------------------------------------------
def discover_pairs(roots):
    pairs, seen, unlabeled = [], set(), 0
    for sroot in roots:
        for dp, dn, _ in os.walk(sroot):
            dn[:] = [d for d in dn if d.lower() not in IGNORE_DIRS
                     and os.path.join(dp, d) not in SKIP_PATHS]
            cur = Path(dp)
            if cur.name.lower() != "images":
                continue
            dn[:] = []                                   # do not walk inside images/
            real = str(cur.resolve())
            if real in seen:
                continue
            seen.add(real)

            lab = find_child_dir(cur.parent, "labels")
            if lab is None:
                unlabeled += 1
                continue

            names = find_names_up(cur.parent)
            subs = [d for d in cur.iterdir() if d.is_dir() and normalize_split(d.name)]
            parent_split = normalize_split(cur.parent.name)

            def add(im, lb, split, rec):
                n = count_images(im, recursive=rec)
                if n:
                    pairs.append({"images": Path(os.path.abspath(im)),
                                  "labels": Path(os.path.abspath(lb)),
                                  "split": split, "recursive": rec,
                                  "count": n, "names": names})

            if subs:                                     # Root/images/train + Root/labels/train
                for d in subs:
                    s = normalize_split(d.name)
                    ld = find_split_dir(lab, s)
                    if ld is not None:
                        add(d, ld, s, True)
                add(cur, lab, None, False)               # loose images
            else:                                        # Root/train/images  or  Root/images
                add(cur, lab, parent_split, True)
    if unlabeled:
        print(f"  (skipped {unlabeled} 'images' folders without sibling 'labels')")
    return pairs

_DEFAULT_CACHE = {}
def default_pairs():
    if "pairs" not in _DEFAULT_CACHE:
        roots = []
        for r in ["/kaggle/input", "/content", "/mnt/data", "/workspace", "/data"]:
            if Path(r).is_dir():
                roots.append(Path(r).resolve())
        roots.append(Path.cwd().resolve())
        roots = list(dict.fromkeys(roots))
        print("Search roots:", [str(r) for r in roots])
        _DEFAULT_CACHE["pairs"] = discover_pairs(roots)
    return _DEFAULT_CACHE["pairs"]

def get_pool_pairs(pool):
    override = ALL_PATH_ if pool == "all" else BD_PATH_
    print(f"{pool.upper()} override path:", override)

    if override:
        p = Path(override)
        if not p.exists():
            raise FileNotFoundError(f"{pool.upper()} path does not exist: {p}")
        return discover_pairs([p.resolve()])

    out = []
    for pr in default_pairs():
        s = str(pr["images"]).lower()
        is_bd = BD_HINT_.lower() in s
        if pool == "bd" and is_bd:
            out.append(pr)
        elif pool == "all" and PATH_HINT_.lower() in s and not is_bd:
            out.append(pr)

    # Fallback: hint matched nothing -> use every labelled dataset that is not BD
    if pool == "all" and not out:
        out = [pr for pr in default_pairs() if BD_HINT_.lower() not in str(pr["images"]).lower()]
        if out:
            print("PATH_HINT matched nothing -> using all non-BD datasets found")
    return out

# ------------------------------------------------------------
# class remap (RDD D00/D10/D20 -> Crack, D40 -> Pothole)
# ------------------------------------------------------------
def name_to_new(n):
    s = str(n).strip().lower()
    if "pothole" in s or s == "d40":
        return 1
    if "crack" in s or s in {"d00", "d10", "d20"}:
        return 0
    return None                                          # other damage types are dropped

_MAP_CACHE = {}
def get_class_map(names):
    if MANUAL_CLASS_MAP_ is not None:
        return dict(MANUAL_CLASS_MAP_), "manual"
    key = tuple(names) if names else None
    if key not in _MAP_CACHE:
        if not names:
            m, how = {0: 0, 1: 1}, "identity (no yaml names)"
        else:
            m = {i: name_to_new(n) for i, n in enumerate(names)}
            if all(v is None for v in m.values()):
                m, how = {0: 0, 1: 1}, "identity (names not recognised)"
            else:
                how = "by name"
        _MAP_CACHE[key] = (m, how)
    return _MAP_CACHE[key]

def convert_label(path, cmap):
    """Validate + remap one label file. Returns (lines, dropped_instances)."""
    out, dropped = [], 0
    for line in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines():
        v = line.split()
        if not v:
            continue
        if len(v) != 5:
            raise ValueError(f"expected 5 values, got {len(v)}")
        cf = float(v[0])
        if not np.isfinite(cf) or cf != int(cf) or cf < 0:
            raise ValueError(f"bad class id {v[0]}")
        old = int(cf)
        if old not in cmap:
            raise ValueError(f"class id {old} not defined")
        x, y, w, h = map(float, v[1:])
        if not all(np.isfinite(t) for t in (x, y, w, h)):
            raise ValueError("non-finite coordinates")
        if (x < -COORD_TOL or x > 1 + COORD_TOL or y < -COORD_TOL or y > 1 + COORD_TOL
                or w <= 0 or h <= 0 or w > 1 + COORD_TOL or h > 1 + COORD_TOL):
            raise ValueError(f"box out of range: {x},{y},{w},{h}")
        new = cmap[old]
        if new is None:
            dropped += 1
            continue
        x, y = min(max(x, 0.0), 1.0), min(max(y, 0.0), 1.0)
        w, h = min(w, 1.0), min(h, 1.0)
        out.append(f"{new} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    return out, dropped

def scan_raw_class_ids(pairs, limit=3000):
    """Quick look at which class ids exist in the raw labels (sample)."""
    ids, n = Counter(), 0
    for pr in pairs:
        for lf in pr["labels"].rglob("*.txt"):
            try:
                for line in lf.read_text(encoding="utf-8", errors="ignore").splitlines():
                    v = line.split()
                    if v:
                        ids[v[0]] += 1
            except OSError:
                pass
            n += 1
            if n >= limit:
                return ids
    return ids

# ------------------------------------------------------------
# collect / validate / split / materialise
# ------------------------------------------------------------
def collect_records(pairs):
    out, seen = {s: [] for s in SPLITS}, set()
    for pr in pairs:
        it = pr["images"].rglob("*") if pr["recursive"] else pr["images"].iterdir()
        for img in sorted(it):
            if not is_image(img):
                continue
            key = str(img)
            if key in seen:
                continue
            seen.add(key)
            rel = img.relative_to(pr["images"])
            lab = pr["labels"] / rel.with_suffix(".txt")
            if not lab.is_file():
                lab = pr["labels"] / (img.stem + ".txt")
            out[pr["split"] or "train"].append(
                {"image": img, "label": lab if lab.is_file() else None, "names": pr["names"]})
    return out

def process_records(records):
    clean = {s: [] for s in SPLITS}
    st = {"missing": 0, "empty": 0, "dropped_instances": 0, "invalid": [], "remap": Counter()}
    for s in SPLITS:
        for rec in records[s]:
            cmap, how = get_class_map(rec["names"])
            st["remap"][how + " | " + str(rec["names"])] += 1
            if rec["label"] is None:
                st["missing"] += 1
                if not TREAT_MISSING_AS_BACKGROUND:
                    st["invalid"].append((str(rec["image"]), "missing label"))
                    continue
                rec["lines"] = []
            else:
                try:
                    rec["lines"], d = convert_label(rec["label"], cmap)
                    st["dropped_instances"] += d
                except Exception as e:
                    st["invalid"].append((str(rec["image"]), str(e)))
                    if not DROP_INVALID:
                        raise RuntimeError(f"Invalid label: {rec['label']} -> {e}")
                    continue
                if not rec["lines"]:
                    st["empty"] += 1
            clean[s].append(rec)
    return clean, st

def automatic_split(recs):
    train, valid, test = (list(recs[s]) for s in SPLITS)
    key = lambda r: str(r["image"])
    rng = random.Random(RANDOM_SEED)
    if not valid and not test:                                   # fully unsplit
        train.sort(key=key); rng.shuffle(train)
        n = len(train)
        n_tr, n_va = int(n * TRAIN_RATIO), int(n * VAL_RATIO)
        if n >= 3:
            n_va = max(1, n_va)
            if n - n_tr - n_va < 1:
                n_tr = n - n_va - 1
        return train[:n_tr], train[n_tr:n_tr + n_va], train[n_tr + n_va:]
    if not valid and len(train) > 1:
        train.sort(key=key); rng.shuffle(train)
        k = max(1, int(len(train) * 0.15)); valid, train = train[:k], train[k:]
    if not test and len(train) > 1:
        train.sort(key=key); rng.shuffle(train)
        k = max(1, int(len(train) * 0.10)); test, train = train[:k], train[k:]
    return train, valid, test

def materialise_pool(pool, recs):
    imgs = {s: [] for s in SPLITS}
    cls_cnt = {s: Counter() for s in SPLITS}
    for s in SPLITS:
        idir = YOLO_ROOT / pool / s / "images"
        ldir = YOLO_ROOT / pool / s / "labels"
        idir.mkdir(parents=True, exist_ok=True)
        ldir.mkdir(parents=True, exist_ok=True)
        for i, rec in enumerate(sorted(recs[s], key=lambda r: str(r["image"]))):
            img = rec["image"]
            name = f"{i:06d}_{img.stem}"
            dst_i = idir / (name + img.suffix.lower())
            dst_l = ldir / (name + ".txt")
            try:
                os.symlink(os.path.abspath(img), dst_i)
            except OSError:
                shutil.copy2(img, dst_i)
            dst_l.write_text("\n".join(rec["lines"]) + ("\n" if rec["lines"] else ""))
            for ln in rec["lines"]:
                cls_cnt[s][int(ln.split()[0])] += 1
            imgs[s].append(dst_i)
    return imgs, cls_cnt

# ------------------------------------------------------------
# BUILD
# ------------------------------------------------------------
if YOLO_ROOT.exists():
    shutil.rmtree(YOLO_ROOT)

POOL_IMGS, POOL_SUMMARY = {}, {}

for pool in POOLS_NEEDED:
    print("\n" + "-" * 75)
    print(f"POOL: {pool.upper()}")
    print("-" * 75)
    pairs = get_pool_pairs(pool)
    if not pairs:
        found = sorted({str(p["images"]) for p in default_pairs()})[:15]
        raise FileNotFoundError(
            f"No labelled YOLO folders found for pool '{pool}'.\n"
            f"Set ALL_PATH / BD_PATH in Cell 2 (and RUN Cell 2 again).\n"
            f"Image folders seen (first 15): {found}")

    print("Source folders:", len(pairs))
    for p in pairs[:8]:
        print("  ", p["images"], f"({p['count']} imgs, split={p['split']})")
    if len(pairs) > 8:
        print("   ...")

    raw_ids = scan_raw_class_ids(pairs)
    print("Raw class ids in labels (sample):", dict(sorted(raw_ids.items())))
    print("yaml names found:", pairs[0]["names"])

    records = collect_records(pairs)
    official = {s: len(records[s]) for s in SPLITS}
    print("Official split sizes:", official)

    records, st = process_records(records)
    total = sum(len(records[s]) for s in SPLITS) + len(st["invalid"])
    if st["invalid"] and len(st["invalid"]) > 0.01 * max(total, 1):
        raise RuntimeError(
            f"{len(st['invalid'])}/{total} images have invalid labels. Example: {st['invalid'][0]}\n"
            f"Raw class ids seen: {dict(sorted(raw_ids.items()))}\n"
            "Fix: set MANUAL_CLASS_MAP in Cell 2, e.g. {0:0, 1:0, 2:0, 3:1} "
            "(D00,D10,D20 -> Crack ; D40 -> Pothole), then re-run Cell 2 and Cell 3.")

    tr, va, te = automatic_split(records)
    final = {"train": tr, "valid": va, "test": te}
    for s in SPLITS:
        if not final[s]:
            raise RuntimeError(f"Pool '{pool}': {s} split is empty!")

    POOL_IMGS[pool], cls_cnt = materialise_pool(pool, final)

    print("Remap used:")
    for k, v in st["remap"].most_common(5):
        print(f"   {v:>7} imgs  {k}")
    print("Final split sizes:", {s: len(POOL_IMGS[pool][s]) for s in SPLITS})
    for s in SPLITS:
        print(f"   {s:<6} Crack={cls_cnt[s][0]:>7,}  Pothole={cls_cnt[s][1]:>7,}")
    print(f"Missing labels (-> background): {st['missing']} | empty: {st['empty']} | "
          f"invalid dropped: {len(st['invalid'])} | other-class instances dropped: {st['dropped_instances']}")

    POOL_SUMMARY[pool] = {
        "source_folders": [str(p["images"]) for p in pairs],
        "official_split": official,
        "final_split": {s: len(POOL_IMGS[pool][s]) for s in SPLITS},
        "instances": {s: {"Crack": int(cls_cnt[s][0]), "Pothole": int(cls_cnt[s][1])} for s in SPLITS},
        "missing_labels": st["missing"], "empty_labels": st["empty"],
        "invalid_dropped": len(st["invalid"]),
        "other_class_instances_dropped": st["dropped_instances"],
        "remap": {k: v for k, v in st["remap"].items()},
    }

# ------------------------------------------------------------
# EFFECTIVE SPLITS FOR THIS EXPERIMENT
# ------------------------------------------------------------
split_imgs = {s: [p for pool in EFFECTIVE[s] for p in POOL_IMGS[pool][s]] for s in SPLITS}
split_dirs = {s: [str(YOLO_ROOT / pool / s / "images") for pool in EFFECTIVE[s]] for s in SPLITS}

all_imgs = POOL_IMGS.get("all", {s: [] for s in SPLITS})
bd_imgs = POOL_IMGS.get("bd", {s: [] for s in SPLITS})

experiment_name = {1: "All -> All", 2: "BD -> BD", 3: "All-model -> BD (eval only)",
                   4: "All+BD -> BD"}[EXPERIMENT]

print("\n" + "=" * 75)
print("FINAL CONFIGURATION | Experiment", EXPERIMENT, ":", experiment_name)
print("=" * 75)
for s in SPLITS:
    print(f"{s.upper():<6}: {len(split_imgs[s]):>8,} images  <- pools {EFFECTIVE[s]}")
if EXPERIMENT == 3:
    print("(train split is unused: Exp 3 only evaluates an existing model)")
print("Classes:", NUM_CLASSES, CLASS_NAMES)
print("YOLO_ROOT:", YOLO_ROOT)
print("\nCELL 3 COMPLETED SUCCESSFULLY")

DATASET DISCOVERY | EXPERIMENT 1 | pools: ['all']
ALL_PATH: None
BD_PATH : None

---------------------------------------------------------------------------
POOL: ALL
---------------------------------------------------------------------------
ALL override path: None
Search roots: ['/kaggle/input', '/content', '/kaggle/working']


Source folders: 5
   /kaggle/input/datasets/mehedihasanfoysal/roaddamageallcountrydata/RoadDamageAllCountryData/RDD2022_YOLO/China/images (4378 imgs, split=None)
   /kaggle/input/datasets/mehedihasanfoysal/roaddamageallcountrydata/RoadDamageAllCountryData/RDD2022_YOLO/Czech/images (2829 imgs, split=None)
   /kaggle/input/datasets/mehedihasanfoysal/roaddamageallcountrydata/RoadDamageAllCountryData/RDD2022_YOLO/India/images (7706 imgs, split=None)
   /kaggle/input/datasets/mehedihasanfoysal/roaddamageallcountrydata/RoadDamageAllCountryData/RDD2022_YOLO/Japan/images (10506 imgs, split=None)
   /kaggle/input/datasets/mehedihasanfoysal/roaddamageallcountrydata/RoadDamageAllCountryData/RDD2022_YOLO/United_States/images (4805 imgs, split=None)


Raw class ids in labels (sample): {'0': 4332, '1': 158}
yaml names found: None


Official split sizes: {'train': 30224, 'valid': 0, 'test': 0}


Remap used:
     30224 imgs  identity (no yaml names) | None
Final split sizes: {'train': 21156, 'valid': 6044, 'test': 3023}
   train  Crack= 26,305  Pothole=  4,200
   valid  Crack=  7,579  Pothole=  1,206
   test   Crack=  3,807  Pothole=    677
Missing labels (-> background): 0 | empty: 9371 | invalid dropped: 1 | other-class instances dropped: 0

FINAL CONFIGURATION | Experiment 1 : All -> All
TRAIN :   21,156 images  <- pools ['all']
VALID :    6,044 images  <- pools ['all']
TEST  :    3,023 images  <- pools ['all']
Classes: 2 ['Crack', 'Pothole']
YOLO_ROOT: /kaggle/working/auto_dataset/yolo_dataset

CELL 3 COMPLETED SUCCESSFULLY


In [4]:
# ============================================================
# CELL 4: LABEL VERIFICATION (on the built dataset)
# Bad files are removed from YOLO_ROOT (symlink + label only)
# ============================================================
from collections import Counter
from pathlib import Path
import os
import numpy as np
import pandas as pd

MAX_BAD_RATIO = 0.001   # 0.1%

def label_of(p):
    """YOLO_ROOT/<pool>/<split>/images/x.jpg -> .../labels/x.txt"""
    p = Path(p)
    return p.parent.parent / "labels" / (p.stem + ".txt")

def parse_line(line):
    t = line.split()
    if len(t) != 5:
        raise ValueError(f"Expected 5 values, got {len(t)}")
    kf = float(t[0])
    if not np.isfinite(kf) or not kf.is_integer():
        raise ValueError(f"Invalid class ID: {t[0]}")
    k = int(kf)
    if not (0 <= k < NUM_CLASSES):
        raise ValueError(f"Class ID out of range: {k}")
    x, y, w, h = map(float, t[1:])
    if not all(np.isfinite(v) for v in (x, y, w, h)):
        raise ValueError("Non-finite coordinates")
    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
        raise ValueError(f"Box out of range: x={x}, y={y}, w={w}, h={h}")
    return k

def verify(split):
    missing, bad_records, bad_images, cnt = 0, [], set(), Counter()
    images = split_imgs.get(split, [])
    for p in images:
        p = Path(p)
        lp = label_of(p)
        if not lp.is_file():
            missing += 1
            continue
        file_counts, file_bad = Counter(), False
        for n, line in enumerate(lp.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
            if not line.strip():
                continue
            try:
                file_counts[parse_line(line)] += 1
            except Exception as e:
                file_bad = True
                bad_records.append({"split": split, "label_file": str(lp), "image": str(p),
                                    "line_no": n, "reason": str(e), "line": line[:200]})
        if file_bad:
            bad_images.add(str(p))
        else:
            cnt.update(file_counts)
    return ({"split": split, "images": len(images), "missing_label": missing,
             "bad_label_files": len(bad_images),
             "Crack": cnt[0], "Pothole": cnt[1]}, bad_records, bad_images)

rows, bad_records_all, bad_all = [], [], set()
for s in ("train", "valid", "test"):
    r, recs, b = verify(s)
    rows.append(r); bad_records_all.extend(recs); bad_all |= b

V = pd.DataFrame(rows)
tot = V["Crack"] + V["Pothole"]
V["Crack%"] = np.where(tot > 0, (V["Crack"] / tot * 100).round(1), 0.0)
V["Pothole%"] = np.where(tot > 0, (V["Pothole"] / tot * 100).round(1), 0.0)

print("=" * 65); print("DATASET VERIFICATION REPORT"); print("=" * 65)
display(V)
print("Total images  :", V["images"].sum())
print("Total Crack   :", V["Crack"].sum())
print("Total Pothole :", V["Pothole"].sum())
print("Missing labels:", V["missing_label"].sum())
print("Bad files     :", V["bad_label_files"].sum())

PROJECT = Path(PROJECT); PROJECT.mkdir(parents=True, exist_ok=True)
V.to_csv(PROJECT / "dataset_verification.csv", index=False)
if bad_records_all:
    pd.DataFrame(bad_records_all).to_csv(PROJECT / "bad_label_files.csv", index=False)
    for rec in bad_records_all[:10]:
        print(f" - [{rec['split']}] {Path(rec['label_file']).name} line {rec['line_no']}: "
              f"{rec['reason']} | {rec['line']!r}")

bad_ratio = len(bad_all) / max(int(V["images"].sum()), 1)
assert bad_ratio <= MAX_BAD_RATIO, f"Too many invalid label files ({bad_ratio:.4%})."

# remove bad samples from the working dataset (our own folder only)
for ip in bad_all:
    for f in (Path(ip), label_of(ip)):
        try:
            if os.path.lexists(f):
                os.remove(f)
        except OSError:
            pass
if bad_all:
    split_imgs = {s: [p for p in split_imgs[s] if str(p) not in bad_all] for s in split_imgs}
    print(f"\nRemoved {len(bad_all)} bad samples ({bad_ratio:.4%}).")
split_imgs_clean = split_imgs      # alias for compatibility

print("\nFinal sizes:", {s: len(split_imgs[s]) for s in ("train", "valid", "test")})
print("\nCELL 4 COMPLETED SUCCESSFULLY")

DATASET VERIFICATION REPORT


,split,images,missing_label,bad_label_files,Crack,Pothole,Crack%,Pothole%
0,train,21156,0,0,26305,4200,86.2,13.8
1,valid,6044,0,0,7579,1206,86.3,13.7
2,test,3023,0,0,3807,677,84.9,15.1


Total images  : 30223
Total Crack   : 37691
Total Pothole : 6083
Missing labels: 0
Bad files     : 0

Final sizes: {'train': 21156, 'valid': 6044, 'test': 3023}

CELL 4 COMPLETED SUCCESSFULLY


In [5]:
# ============================================================
# CELL 5: DATA YAML (directory based, no .txt lists)
# ============================================================
from pathlib import Path
import yaml

PROJECT = Path(PROJECT); PROJECT.mkdir(parents=True, exist_ok=True)

def entry(dirs):
    return dirs[0] if len(dirs) == 1 else list(dirs)

for s in ("valid", "test"):
    assert split_dirs[s] and len(split_imgs[s]) > 0, f"{s} split is empty!"
if EXPERIMENT != 3:
    assert split_dirs["train"] and len(split_imgs["train"]) > 0, "Training split is empty!"

DATA_YAML = PROJECT / f"rdd_exp{EXPERIMENT}.yaml"

cfg = {
    "path": str(YOLO_ROOT),
    # Exp 3 never trains, but Ultralytics still wants a 'train' key -> use valid as placeholder
    "train": entry(split_dirs["train"] if EXPERIMENT != 3 else split_dirs["valid"]),
    "val": entry(split_dirs["valid"]),
    "test": entry(split_dirs["test"]),
    "nc": NUM_CLASSES,
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("=" * 60); print("YOLO DATASET CONFIGURATION"); print("=" * 60)
print(DATA_YAML.read_text(encoding="utf-8"))
for s in ("train", "valid", "test"):
    print(f"{s:<6}: {len(split_imgs[s]):>8,} images")
print("\nExperiment:", EXPERIMENT, "|", experiment_name)
print("YAML:", DATA_YAML)
print("\nCell 5 completed successfully!")

YOLO DATASET CONFIGURATION
path: /kaggle/working/auto_dataset/yolo_dataset
train: /kaggle/working/auto_dataset/yolo_dataset/all/train/images
val: /kaggle/working/auto_dataset/yolo_dataset/all/valid/images
test: /kaggle/working/auto_dataset/yolo_dataset/all/test/images
nc: 2
names:
  0: Crack
  1: Pothole

train :   21,156 images
valid :    6,044 images
test  :    3,023 images

Experiment: 1 | All -> All
YAML: /kaggle/working/RDD_Baseline_Run/rdd_exp1.yaml

Cell 5 completed successfully!


In [6]:
# ============================================================
# CELL 6: TRAINING (Exp 1, 2, 4) / LOAD CHECKPOINT (Exp 3)
# ============================================================
import os, gc, time
from pathlib import Path
import torch
from ultralytics import YOLO

def on_epoch_end(trainer):
    # NOTE: with 2-GPU DDP, training runs in a subprocess, so this print may not appear.
    if int(os.environ.get("RANK", -1)) not in (-1, 0):
        return
    m = trainer.metrics or {}
    p, r = m.get("metrics/precision(B)"), m.get("metrics/recall(B)")
    m50, m5095 = m.get("metrics/mAP50(B)"), m.get("metrics/mAP50-95(B)")
    if p is None or r is None or m50 is None:
        return
    f1 = 2 * p * r / (p + r + 1e-9)
    print(f"\n  >>> Epoch {trainer.epoch + 1}/{trainer.epochs} | P: {p:.4f} | R: {r:.4f} | "
          f"F1: {f1:.4f} | mAP50: {m50:.4f} | mAP50-95: {(m5095 or 0.0):.4f}", flush=True)

if EXPERIMENT != 3:
    t0 = time.time()
    model = YOLO(MODEL_WEIGHTS)
    model.add_callback("on_fit_epoch_end", on_epoch_end)

    print("Starting YOLO training | Exp", EXPERIMENT, "|", experiment_name)
    print("Weights:", MODEL_WEIGHTS, "| Data:", DATA_YAML)
    print("Device:", DEVICE, "| Batch:", BATCH)

    model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
        workers=4, patience=PATIENCE, seed=SEED, plots=False,
        project=str(PROJECT), name=RUN_NAME, exist_ok=True,
    )

    RUN_DIR = Path(model.trainer.save_dir)          # actual run folder
    BEST = RUN_DIR / "weights" / "best.pt"
    if not BEST.exists():
        BEST = Path(model.trainer.best)
    assert BEST.exists(), f"best.pt not found. Check {RUN_DIR}"
    print(f"\nTraining time: {(time.time() - t0) / 3600:.2f} hours")

    del model; gc.collect(); torch.cuda.empty_cache()

else:
    BEST = Path(EXP1_BEST) if EXP1_BEST else PROJECT / RUN_NAMES[1] / "weights" / "best.pt"
    assert BEST.exists(), ("EXP 3 needs the Exp 1 best.pt. Set EXP1_BEST in Cell 2 "
                           f"(also looked for {BEST}).")
    print("EXP 3: training skipped. Using checkpoint:", BEST)

BEST = Path(BEST).resolve()
print("\n" + "=" * 65)
print("BEST CHECKPOINT:", BEST, "| EXISTS:", BEST.exists())
print("=" * 65)
print("\nCell 6 completed successfully.")

Starting YOLO training | Exp 1 | All -> All
Weights: yolo11s.pt | Data: /kaggle/working/RDD_Baseline_Run/rdd_exp1.yaml
Device: 0,1 | Batch: 32


Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/RDD_Baseline_Run/rdd_exp1.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_all_country, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspect

Overriding model.yaml nc=80 with nc=2



                   from  n    params  module                                       arguments                     


  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 


... (17,521 outputs left out)

      24/80      5.07G      1.689      1.747      1.577         43        640: 82% ━━━━━━━━━╸── 547/662 3.9it/s 2:34<29.6s

      24/80      5.07G      1.689      1.746      1.577         40        640: 82% ━━━━━━━━━╸── 548/662 3.6it/s 2:35<31.5s

      24/80      5.07G      1.689      1.746      1.577         34        640: 82% ━━━━━━━━━╸── 549/662 4.0it/s 2:35<28.3s

      24/80      5.07G      1.689      1.746      1.577         32        640: 83% ━━━━━━━━━╸── 550/662 3.8it/s 2:35<29.8s

In [ ]:
# ============================================================
# CELL 7: FINAL EVALUATION (valid + test)
# ============================================================
import numpy as np, pandas as pd
from pathlib import Path
from ultralytics import YOLO

best = YOLO(str(BEST))
PROJECT = Path(PROJECT); PROJECT.mkdir(parents=True, exist_ok=True)

model_names = [str(best.names[i]).lower() for i in sorted(best.names)]
if model_names != [c.lower() for c in CLASS_NAMES]:
    print("WARNING: model class names", model_names, "differ from", CLASS_NAMES)

print("=" * 65); print("FINAL MODEL EVALUATION"); print("=" * 65)
print("Experiment:", EXPERIMENT, "|", experiment_name, "| Run:", RUN_NAME)
print("Checkpoint:", BEST)

def collect(r, split):
    idx = {int(c): i for i, c in enumerate(r.box.ap_class_index)}
    P, R = float(r.box.mp), float(r.box.mr)
    rows = [dict(split=split, cls="ALL", P=P, R=R, F1=2 * P * R / (P + R + 1e-9),
                 mAP50=float(r.box.map50), mAP50_95=float(r.box.map))]
    for c, name in enumerate(CLASS_NAMES):
        if c not in idx:
            continue
        i = idx[c]
        p, rc = float(r.box.p[i]), float(r.box.r[i])
        rows.append(dict(split=split, cls=name, P=p, R=rc, F1=2 * p * rc / (p + rc + 1e-9),
                         mAP50=float(r.box.ap50[i]), mAP50_95=float(np.mean(r.box.ap[i]))))
    return rows

final_rows, eval_dirs = [], {}
for split_arg, tag in (("val", "valid"), ("test", "test")):
    print("\n" + "=" * 65); print(f"Evaluating {tag.upper()}..."); print("=" * 65)
    r = best.val(data=str(DATA_YAML), split=split_arg, imgsz=IMGSZ, batch=VAL_BATCH,
                 conf=0.001, iou=0.7, device=VAL_DEVICE, plots=True, verbose=False,
                 project=str(PROJECT), name=f"final_{tag}", exist_ok=True)
    final_rows.extend(collect(r, tag))
    eval_dirs[tag] = Path(r.save_dir)

FINAL = pd.DataFrame(final_rows)[["split", "cls", "P", "R", "F1", "mAP50", "mAP50_95"]]
print("\n" + "=" * 85); print("FINAL EVALUATION RESULTS"); print("=" * 85)
display(FINAL.round(4))
FINAL.to_csv(PROJECT / "final_results.csv", index=False, float_format="%.6f")

t = FINAL[(FINAL["split"] == "test") & (FINAL["cls"] == "ALL")].iloc[0]
print("\n" + "=" * 65)
print(f"FINAL TEST RESULT (Exp {EXPERIMENT}: {RUN_NAME})  <- report this one")
print("=" * 65)
print(f"Precision : {t.P:.4f}\nRecall    : {t.R:.4f}\nF1-Score  : {t.F1:.4f}")
print(f"mAP50     : {t.mAP50:.4f}\nmAP50-95  : {t.mAP50_95:.4f}")
print("\nValid dir:", eval_dirs["valid"], "\nTest dir :", eval_dirs["test"])
print("\nCell 7 done.")

In [ ]:
# ============================================================
# CELL 8: CONFUSION MATRIX (valid + test)
# Ultralytics layout: matrix[predicted, true]; last index = background
# Normalised per TRUE class (column) including missed detections,
# so the diagonal equals recall at CM_CONF.
# ============================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

best = YOLO(str(BEST))
PROJECT = Path(PROJECT); PROJECT.mkdir(parents=True, exist_ok=True)
NC = NUM_CLASSES

def plot_cm(norm, counts, title, path):
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(norm, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(NC)); ax.set_yticks(range(NC))
    ax.set_xticklabels(CLASS_NAMES, fontsize=11); ax.set_yticklabels(CLASS_NAMES, fontsize=11)
    ax.set_xlabel("True Class", fontsize=12); ax.set_ylabel("Predicted Class", fontsize=12)
    ax.set_title(title, fontsize=13, pad=15)
    for i in range(NC):
        for j in range(NC):
            ax.text(j, i, f"{norm[i, j]:.3f}\n(n={int(counts[i, j])})", ha="center", va="center",
                    color="white" if norm[i, j] > 0.5 else "black", fontsize=11, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Proportion of true class")
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

cm_paths, cm_results, cm_normalized = {}, {}, {}
bg = ["Background"]

for split_arg, tag in (("val", "valid"), ("test", "test")):
    print("\n" + "=" * 65); print(f"{tag.upper()} CONFUSION MATRIX (conf={CM_CONF})"); print("=" * 65)
    r = best.val(data=str(DATA_YAML), split=split_arg, imgsz=IMGSZ, batch=VAL_BATCH,
                 conf=CM_CONF, iou=0.7, device=VAL_DEVICE, plots=True, verbose=False,
                 project=str(PROJECT), name=f"cm_{tag}", exist_ok=True)

    cmo = getattr(r, "confusion_matrix", None)
    assert cmo is not None, "Confusion matrix not available in this Ultralytics version."
    full = np.asarray(cmo.matrix, dtype=float)
    assert full.shape == (NC + 1, NC + 1), f"Unexpected shape {full.shape}"

    full_df = pd.DataFrame(full.astype(int),
                           index=[f"Pred_{c}" for c in CLASS_NAMES + bg],
                           columns=[f"True_{c}" for c in CLASS_NAMES + bg])
    print("Full matrix (with background):"); display(full_df)
    full_df.to_csv(PROJECT / f"confusion_matrix_full_{tag}.csv")

    counts = full[:NC, :NC]
    col_sum = full[:, :NC].sum(axis=0, keepdims=True)     # GT per class incl. missed
    norm = np.divide(counts, col_sum, out=np.zeros_like(counts), where=col_sum != 0)

    idx = [f"Pred_{c}" for c in CLASS_NAMES]; col = [f"True_{c}" for c in CLASS_NAMES]
    cm_df = pd.DataFrame(counts.astype(int), index=idx, columns=col)
    print("Class-only raw matrix:"); display(cm_df)
    cm_df.to_csv(PROJECT / f"confusion_matrix_{tag}.csv")
    pd.DataFrame(norm, index=idx, columns=col).to_csv(PROJECT / f"confusion_matrix_normalized_{tag}.csv")

    p = PROJECT / f"confusion_matrix_{tag}.png"
    plot_cm(norm, counts, f"Confusion Matrix ({tag.upper()}, conf={CM_CONF})", p)
    cm_paths[tag], cm_results[tag], cm_normalized[tag] = p, counts, norm

print("\n" + "=" * 65); print("CONFUSION MATRICES COMPLETED"); print("=" * 65)
for tag in ("valid", "test"):
    print(f"\n{tag.upper()} raw:\n{cm_results[tag].astype(int)}")
    print(f"{tag.upper()} normalized:\n{np.round(cm_normalized[tag], 4)}")
print("\nCell 8 done!")

In [ ]:
# ============================================================
# CELL 9: TRAINING CURVES & VALID-vs-TEST COMPARISON
# ============================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image as IPImage

PROJECT = Path(PROJECT); PROJECT.mkdir(parents=True, exist_ok=True)
print("=" * 70); print(f"EXPERIMENT {EXPERIMENT}: VISUALIZATION"); print("=" * 70)

best_ep, curve_path = None, None

# ---------------- training curves ----------------
if EXPERIMENT != 3:
    results_file = Path(RUN_DIR) / "results.csv"
    assert results_file.exists(), f"results.csv not found: {results_file}"
    df = pd.read_csv(results_file)
    df.columns = df.columns.str.strip()

    Pc, Rc = "metrics/precision(B)", "metrics/recall(B)"
    M50, M5095 = "metrics/mAP50(B)", "metrics/mAP50-95(B)"
    need = ["epoch", Pc, Rc, M50, M5095, "train/box_loss", "val/box_loss",
            "train/cls_loss", "val/cls_loss", "train/dfl_loss", "val/dfl_loss"]
    miss = [c for c in need if c not in df.columns]
    assert not miss, f"Missing columns in results.csv: {miss}"

    df["F1"] = 2 * df[Pc] * df[Rc] / (df[Pc] + df[Rc] + 1e-9)
    df["fitness"] = 0.1 * df[M50] + 0.9 * df[M5095]          # Ultralytics best.pt criterion
    best_idx = df["fitness"].idxmax()
    best_ep = int(df.loc[best_idx, "epoch"])
    print("Best epoch:", best_ep)
    print(df.loc[best_idx, [Pc, Rc, "F1", M50, M5095]].round(4))

    panels = [("Box Loss", ["train/box_loss", "val/box_loss"]),
              ("Classification Loss", ["train/cls_loss", "val/cls_loss"]),
              ("DFL Loss", ["train/dfl_loss", "val/dfl_loss"]),
              ("Precision / Recall", [Pc, Rc]),
              ("F1 Score", ["F1"]),
              ("mAP50 / mAP50-95", [M50, M5095])]
    fig, axes = plt.subplots(2, 3, figsize=(19, 11))
    for ax, (title, cols) in zip(axes.ravel(), panels):
        for c in cols:
            ax.plot(df["epoch"], df[c], label=c.split("/")[-1], linewidth=2)
        ax.axvline(best_ep, linestyle="--", linewidth=1.5, color="gray", label=f"Best epoch ({best_ep})")
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.suptitle(f"Training Dynamics - Experiment {EXPERIMENT}", fontsize=16, fontweight="bold")
    fig.tight_layout()
    curve_path = PROJECT / "training_curves.png"
    fig.savefig(curve_path, dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)
    df.to_csv(PROJECT / "training_history.csv", index=False)
else:
    print("EXP 3: no training curves (training skipped).")

# ---------------- valid vs test ----------------
assert "FINAL" in globals(), "Run Cell 7 first."
FP = FINAL.copy(); FP["cls"] = FP["cls"].astype(str)
plot_classes = [c for c in ["ALL"] + CLASS_NAMES if c in FP["cls"].unique()]
assert {"valid", "test"}.issubset(set(FP["split"])), "valid/test results missing"

metrics = ["P", "R", "F1", "mAP50", "mAP50_95"]
fig, axes = plt.subplots(1, 5, figsize=(25, 6))
for ax, m in zip(axes, metrics):
    pv = (FP.pivot_table(index="cls", columns="split", values=m, aggfunc="first")
            .reindex(plot_classes).reindex(columns=["valid", "test"]))
    pv.plot.bar(ax=ax, rot=0, width=0.75)
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.3f", fontsize=8, padding=2)
    ax.set_title(m, fontsize=12, fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("Score"); ax.set_ylim(0, 1.15)
    ax.grid(axis="y", alpha=0.3); ax.legend(title="Split", fontsize=8)
fig.suptitle(f"Validation vs Test Performance - EXP {EXPERIMENT}", fontsize=16, fontweight="bold")
fig.tight_layout()
bar_path = PROJECT / "final_bar.png"
fig.savefig(bar_path, dpi=300, bbox_inches="tight")
plt.show(); plt.close(fig)

# ---------------- PR curves (test) ----------------
if "eval_dirs" in globals():
    for fn in ("BoxPR_curve.png", "PR_curve.png"):
        f = Path(eval_dirs["test"]) / fn
        if f.exists():
            print("Displaying:", fn)
            display(IPImage(filename=str(f)))

# ---------------- save ----------------
FINAL.to_csv(PROJECT / "final_performance.csv", index=False, float_format="%.6f")
display(FINAL.round(4))

print("\n" + "=" * 70)
print(f"EXPERIMENT {EXPERIMENT} VISUALIZATION COMPLETED")
if best_ep is not None:
    print("Best epoch:", best_ep, "| Curves:", curve_path)
print("Comparison:", bar_path)
print("\nCell 9 completed successfully!")

In [ ]:
# ============================================================
# CELL 10: EXPORT RESULTS (zip)
# ============================================================
import json, shutil, zipfile
from pathlib import Path
import ultralytics

PROJECT = Path(PROJECT)
OUTP = WORK / f"Exp{EXPERIMENT}_Output"
if OUTP.exists():
    shutil.rmtree(OUTP)
OUTP.mkdir(parents=True, exist_ok=True)

print("=" * 70); print(f"EXPORTING EXPERIMENT {EXPERIMENT}"); print("=" * 70)

files = []
if EXPERIMENT != 3:
    files += [Path(RUN_DIR) / "weights" / "best.pt", Path(RUN_DIR) / "weights" / "last.pt",
              Path(RUN_DIR) / "results.csv", Path(RUN_DIR) / "args.yaml",
              PROJECT / "training_history.csv", PROJECT / "training_curves.png"]

files += [PROJECT / "final_results.csv", PROJECT / "final_performance.csv", PROJECT / "final_bar.png",
          PROJECT / "dataset_verification.csv", PROJECT / "bad_label_files.csv", Path(DATA_YAML)]

for tag in ("valid", "test"):
    files += [PROJECT / f"confusion_matrix_{tag}.png", PROJECT / f"confusion_matrix_{tag}.csv",
              PROJECT / f"confusion_matrix_full_{tag}.csv",
              PROJECT / f"confusion_matrix_normalized_{tag}.csv"]

copied, skipped = [], []
for src in files:
    src = Path(src)
    if src.is_file():
        shutil.copy2(src, OUTP / src.name); copied.append(src.name)
    else:
        skipped.append(str(src))

# PR / F1 curves from the evaluation runs
if "eval_dirs" in globals():
    for tag, d in eval_dirs.items():
        for fn in ["BoxPR_curve.png", "PR_curve.png", "BoxF1_curve.png", "BoxP_curve.png",
                   "BoxR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png"]:
            f = Path(d) / fn
            if f.is_file():
                shutil.copy2(f, OUTP / f"{tag}_{fn}"); copied.append(f"{tag}_{fn}")

if EXPERIMENT == 3:
    shutil.copy2(BEST, OUTP / "evaluation_model.pt"); copied.append("evaluation_model.pt")

# experiment info
info = {
    "experiment": int(EXPERIMENT), "description": experiment_name, "run_name": RUN_NAME,
    "initial_weights": str(MODEL_WEIGHTS), "evaluated_checkpoint": str(BEST),
    "exp1_checkpoint_arg": str(EXP1_BEST) if EXP1_BEST else None,
    "dataset_yaml": str(DATA_YAML), "effective_pools": EFFECTIVE,
    "image_size": int(IMGSZ), "batch_size": int(BATCH), "epochs": int(EPOCHS),
    "patience": int(PATIENCE), "device": str(DEVICE), "seed": int(SEED),
    "class_names": list(CLASS_NAMES), "training_skipped": EXPERIMENT == 3,
    "ultralytics": ultralytics.__version__,
    "split_sizes": {s: len(split_imgs[s]) for s in ("train", "valid", "test")},
    "pool_summary": POOL_SUMMARY,
}
with open(OUTP / "experiment_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, indent=4, ensure_ascii=False, default=str)

if "FINAL" in globals():
    row = FINAL[(FINAL["split"] == "test") & (FINAL["cls"] == "ALL")]
    if not row.empty:
        r = row.iloc[0]
        with open(OUTP / "final_test_summary.json", "w", encoding="utf-8") as f:
            json.dump({"experiment": int(EXPERIMENT), "run_name": RUN_NAME,
                       "Precision": float(r["P"]), "Recall": float(r["R"]), "F1": float(r["F1"]),
                       "mAP50": float(r["mAP50"]), "mAP50-95": float(r["mAP50_95"])}, f, indent=4)

# zip + integrity check
zp = WORK / f"Exp{EXPERIMENT}_Output.zip"
if zp.exists():
    zp.unlink()
with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for f in sorted(OUTP.rglob("*")):
        if f.is_file():
            z.write(f, arcname=str(Path(OUTP.name) / f.relative_to(OUTP)))
with zipfile.ZipFile(zp, "r") as z:
    assert z.testzip() is None, "ZIP integrity check failed"
    names = z.namelist()

print("\nEXPORTED:")
for n in copied:
    print("  [OK]", n)
if skipped:
    print("\nNOT FOUND (fine if not applicable):")
    for s in skipped:
        print("  [SKIP]", s)
print("\nTotal files in zip:", len(names))
print("ZIP:", zp, f"({zp.stat().st_size / 1024**2:.2f} MB)")
print("\nZIP INTEGRITY: PASSED\nCELL 10 COMPLETED SUCCESSFULLY")